###### =====================================================================
#### STEP 1: OPTIMIZED DATA LOADING & STRUCTURAL LEFT-MERGE
###### =====================================================================

In [29]:
import numpy as np
import pandas as pd
import os
import gc
from sklearn.preprocessing import OrdinalEncoder


print("Loading datasets...")
# Using a left join to retain 100% of transactions. Missing identity 
# records are naturally preserved as structural null values.
BASE_DIR = '/Users/abannee/Documents/GitHub/fraud_detection_ml/'
DATA_DIR = os.path.join(BASE_DIR, 'data/raw')

print("Loading Train Datasets...")
train_transaction = pd.read_csv(os.path.join(DATA_DIR, 'train_transaction.csv'))
train_identity = pd.read_csv(os.path.join(DATA_DIR, "train_identity.csv"))

print("Merging transaction and identity data on TransactionID...")
df = pd.merge(train_transaction, train_identity, on="TransactionID", how="left")

# Clean up memory allocation from unmerged frames
del train_transaction, train_identity

# Force immediate background RAM reclamation
gc.collect() 


print(f"Data loading complete. data shape: {df.shape}")

Loading datasets...
Loading Train Datasets...
Merging transaction and identity data on TransactionID...
Data loading complete. data shape: (590540, 434)


###### =====================================================================
#### STEP 2: CHRONOLOGICAL SORTING & TIME-BASED SPLITTING (70/30)
###### =====================================================================

In [30]:
print("Sorting data chronologically by TransactionDT to prevent look-ahead bias...")
df = df.sort_values(by="TransactionDT").reset_index(drop=True)

# Calculate deterministic, shuffle-free split index for OOT validation
split_idx = int(len(df) * 0.70)

print(# Using a left join to retain 100% of transactions. Missing identity 
# records are naturally preserved as structural null values.
f"Splitting data: Training on first 70% ({split_idx} rows), Testing on remaining 30% ({len(df) - split_idx} rows)...")
train_df = df.iloc[:split_idx].reset_index(drop=True)
test_df = df.iloc[split_idx:].reset_index(drop=True)

# Separate features and target
X_train = train_df.drop(columns=["isFraud"])
y_train = train_df["isFraud"]
X_test = test_df.drop(columns=["isFraud"])
y_test = test_df["isFraud"]

Sorting data chronologically by TransactionDT to prevent look-ahead bias...
Splitting data: Training on first 70% (413378 rows), Testing on remaining 30% (177162 rows)...


###### =====================================================================
#### STEP 3: DEFENSIVE CATEGORICAL ENCODING
###### =====================================================================

In [31]:
# Define high-cardinality and operational categorical columns to encode
categorical_cols = ["ProductCD", "card4", "id_30", "id_31", "DeviceType"]
# Ensure columns exist in the dataframe before processing
categorical_cols = [col for col in categorical_cols if col in X_train.columns]

# --- A. Safe Label (Ordinal) Encoding ---
print("Executing safe Ordinal Encoding...")
# Configure encoder defensively to route unseen production/test labels to a fallback value (-1)
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

# Fit strictly on training data, transform both partitions
X_train[categorical_cols] = ordinal_encoder.fit_transform(X_train[categorical_cols].astype(str))
X_test[categorical_cols] = ordinal_encoder.transform(X_test[categorical_cols].astype(str))


# --- B. Safe Target Encoding (Out-of-Fold / Smoothed Mock) ---
print("Executing safe Target Encoding pipeline...")

# Calculate global training baseline fraud rate to act as a neutral anchor
global_training_mean = y_train.mean()

for col in categorical_cols:
    # Example Target Encoding Strategy with local calculation.
    # To prevent target leakage completely, we calculate map dictionaries 
    # strictly from training pairs, incorporating a basic frequency map.
    
    # Calculate category stats within training set
    stats = pd.DataFrame({"target": y_train, "label": train_df[col]}).groupby("label")
    category_means = stats["target"].mean()
    category_counts = stats["target"].count()
    
    # Simple m-estimate smoothing formula: (mean * count + global_mean * m) / (count + m)
    m = 10 
    smoothed_vals = (category_means * category_counts + global_training_mean * m) / (category_counts + m)
    encoding_map = smoothed_vals.to_dict()
    
    # Map back to train and test data using training metrics only
    # Defensively fill any novel/unseen categories in test with the neutral global training mean
    X_train[f"{col}_target_enc"] = train_df[col].map(encoding_map).fillna(global_training_mean)
    X_test[f"{col}_target_enc"] = test_df[col].map(encoding_map).fillna(global_training_mean)

print("Data preparation pipeline complete! Ready for feature engineering or modeling steps.")

Executing safe Ordinal Encoding...
Executing safe Target Encoding pipeline...


/var/folders/2w/c8v4yt5j21s3nkv8j9x_fc340000gn/T/ipykernel_61136/3234097378.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[f"{col}_target_enc"] = train_df[col].map(encoding_map).fillna(global_training_mean)
/var/folders/2w/c8v4yt5j21s3nkv8j9x_fc340000gn/T/ipykernel_61136/3234097378.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test[f"{col}_target_enc"] = test_df[col].map(encoding_map).fillna(global_training_mean)
/var/folders/2w/c8v4yt5j21s3nkv8j9x_fc340000gn/T/ipykernel_61136/3234097378.py:39: Performance

Data preparation pipeline complete! Ready for feature engineering or modeling steps.


/var/folders/2w/c8v4yt5j21s3nkv8j9x_fc340000gn/T/ipykernel_61136/3234097378.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[f"{col}_target_enc"] = train_df[col].map(encoding_map).fillna(global_training_mean)
/var/folders/2w/c8v4yt5j21s3nkv8j9x_fc340000gn/T/ipykernel_61136/3234097378.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test[f"{col}_target_enc"] = test_df[col].map(encoding_map).fillna(global_training_mean)


In [ ]:

# STEP 1: OPTIMIZED DATA LOADING & STRUCTURAL LEFT-MERGE¶
import numpy as np
import pandas as pd
import os
import gc
from sklearn.preprocessing import OrdinalEncoder


print("Loading datasets...")
# Using a left join to retain 100% of transactions. Missing identity 
# records are naturally preserved as structural null values.
BASE_DIR = '/Users/abannee/Documents/GitHub/fraud_detection_ml/'
DATA_DIR = os.path.join(BASE_DIR, 'data/raw')

print("Loading Train Datasets...")
train_transaction = pd.read_csv(os.path.join(DATA_DIR, 'train_transaction.csv'))
train_identity = pd.read_csv(os.path.join(DATA_DIR, "train_identity.csv"))

print("Merging transaction and identity data on TransactionID...")
df = pd.merge(train_transaction, train_identity, on="TransactionID", how="left")

# Clean up memory allocation from unmerged frames
del train_transaction, train_identity

# Force immediate background RAM reclamation
gc.collect() 


print(f"Data loading complete. data shape: {df.shape}")



# STEP 2: CHRONOLOGICAL SORTING & TIME-BASED SPLITTING (70/30)¶


print("Sorting data chronologically by TransactionDT to prevent look-ahead bias...")
df = df.sort_values(by="TransactionDT").reset_index(drop=True)

# Calculate deterministic, shuffle-free split index for OOT validation
split_idx = int(len(df) * 0.70)

print(# Using a left join to retain 100% of transactions. Missing identity 
# records are naturally preserved as structural null values.
f"Splitting data: Training on first 70% ({split_idx} rows), Testing on remaining 30% ({len(df) - split_idx} rows)...")
train_df = df.iloc[:split_idx].reset_index(drop=True)
test_df = df.iloc[split_idx:].reset_index(drop=True)

# Separate features and target
X_train = train_df.drop(columns=["isFraud"])
y_train = train_df["isFraud"]
X_test = test_df.drop(columns=["isFraud"])
y_test = test_df["isFraud"]


# STEP 3: DEFENSIVE CATEGORICAL ENCODING


# Define high-cardinality and operational categorical columns to encode
categorical_cols = ["ProductCD", "card4", "id_30", "id_31", "DeviceType"]
# Ensure columns exist in the dataframe before processing
categorical_cols = [col for col in categorical_cols if col in X_train.columns]

# --- A. Safe Label (Ordinal) Encoding ---
print("Executing safe Ordinal Encoding...")
# Configure encoder defensively to route unseen production/test labels to a fallback value (-1)
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

# Fit strictly on training data, transform both partitions
X_train[categorical_cols] = ordinal_encoder.fit_transform(X_train[categorical_cols].astype(str))
X_test[categorical_cols] = ordinal_encoder.transform(X_test[categorical_cols].astype(str))


# --- B. Safe Target Encoding (Out-of-Fold / Smoothed Mock) ---
print("Executing safe Target Encoding pipeline...")

# Calculate global training baseline fraud rate to act as a neutral anchor
global_training_mean = y_train.mean()

for col in categorical_cols:
    # Example Target Encoding Strategy with local calculation.
    # To prevent target leakage completely, we calculate map dictionaries 
    # strictly from training pairs, incorporating a basic frequency map.
    
    # Calculate category stats within training set
    stats = pd.DataFrame({"target": y_train, "label": train_df[col]}).groupby("label")
    category_means = stats["target"].mean()
    category_counts = stats["target"].count()
    
    # Simple m-estimate smoothing formula: (mean * count + global_mean * m) / (count + m)
    m = 10 
    smoothed_vals = (category_means * category_counts + global_training_mean * m) / (category_counts + m)
    encoding_map = smoothed_vals.to_dict()
    
    # Map back to train and test data using training metrics only
    # Defensively fill any novel/unseen categories in test with the neutral global training mean
    X_train[f"{col}_target_enc"] = train_df[col].map(encoding_map).fillna(global_training_mean)
    X_test[f"{col}_target_enc"] = test_df[col].map(encoding_map).fillna(global_training_mean)

print("Data preparation pipeline complete! Ready for feature engineering or modeling steps.")